# Neutrality Test — Streamlit App

This notebook builds and launches an interactive Streamlit app for the
`neutrality_lrt` function from `neutrality_test.py`.

**Requirements**
```
pip install streamlit numpy pandas scipy statsmodels
```

Run all cells in order. The last cell starts the Streamlit server.

## 1 · Install dependencies (run once)

In [ ]:
%pip install streamlit numpy pandas scipy statsmodels --quiet

## 2 · Write `neutrality_test.py` to disk

This cell saves the core module so the Streamlit app can import it.

In [ ]:
neutrality_test_source = '''
"""
Neutrality test for donor→recipient transmission experiments.
"""
from __future__ import annotations
from dataclasses import dataclass
from typing import Iterable, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import gammaln
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

_MIN_PVAL = float(np.nextafter(0.0, 1.0))

def _log_beta(a, b):
    return float(gammaln(a) + gammaln(b) - gammaln(a + b))

def _log_bb_pmf(x, n, a, b):
    return float(
        (gammaln(n + 1) - gammaln(x + 1) - gammaln(n - x + 1))
        + _log_beta(x + a, n - x + b)
        - _log_beta(a, b)
    )

def _loglik_feature(q, x_vec, n_vec, Nb_vec, *, alpha0, q_bounds):
    lo, hi = q_bounds
    q = float(np.clip(q, lo, hi))
    ll = 0.0
    for xm, nm, Nbm in zip(x_vec, n_vec, Nb_vec):
        if nm <= 0:
            continue
        a = float(Nbm * q + alpha0)
        b = float(Nbm * (1.0 - q) + alpha0)
        ll += _log_bb_pmf(int(xm), int(nm), a, b)
    return float(ll)

def _fit_q_mle(x_vec, n_vec, Nb_vec, *, alpha0, q_bounds):
    lo, hi = q_bounds
    def nll(q):
        return -_loglik_feature(q, x_vec, n_vec, Nb_vec, alpha0=alpha0, q_bounds=q_bounds)
    res = minimize_scalar(nll, bounds=(lo, hi), method="bounded")
    return float(res.x), float(-res.fun)

def _as_1d_float_array(x, length):
    if np.isscalar(x):
        return np.full(length, float(x), dtype=float)
    arr = np.asarray(x, dtype=float).reshape(-1)
    if arr.shape[0] != length:
        raise ValueError(f"bottleneck has length {arr.shape[0]}, expected {length}")
    return arr

def neutrality_lrt(
    donor_counts, recipient_counts, bottleneck, *,
    feature_ids=None, pseudocount=1e-6, alpha0=1e-12,
    min_p=1e-8, q_bounds=(1e-12, 1 - 1e-12),
    fdr_alpha=0.05, fdr_method="fdr_bh",
):
    if isinstance(recipient_counts, pd.DataFrame):
        X = recipient_counts.to_numpy(dtype=np.int64)
        cols_from_recip = list(recipient_counts.columns)
    else:
        X = np.asarray(recipient_counts, dtype=np.int64)
        cols_from_recip = None
    if X.ndim == 1:
        X = X.reshape(1, -1)
    if X.ndim != 2:
        raise ValueError("recipient_counts must be 1D or 2D")
    M, K = X.shape

    if isinstance(donor_counts, pd.Series):
        donor_arr = donor_counts.to_numpy(dtype=float).reshape(-1)
        ids_from_donor = list(donor_counts.index)
    else:
        donor_arr = np.asarray(donor_counts, dtype=float).reshape(-1)
        ids_from_donor = None
    if donor_arr.shape[0] != K:
        raise ValueError(f"donor_counts length does not match K={K}")

    if feature_ids is None:
        if cols_from_recip is not None:
            feature_ids = cols_from_recip
        elif ids_from_donor is not None:
            feature_ids = ids_from_donor
        else:
            feature_ids = list(range(K))

    Nb_vec = _as_1d_float_array(bottleneck, M)
    n_tot  = X.sum(axis=1).astype(np.int64)
    donor_adj = donor_arr + float(pseudocount)
    p_donor   = donor_adj / donor_adj.sum()

    q_hat = np.full(K, np.nan)
    LR    = np.full(K, np.nan)
    pval  = np.full(K, np.nan)

    for j in range(K):
        p0 = float(p_donor[j])
        if (not np.isfinite(p0)) or (p0 < min_p) or (p0 > 1.0 - min_p):
            continue
        x_vec = X[:, j].astype(np.int64, copy=False)
        ll0 = _loglik_feature(p0, x_vec, n_tot, Nb_vec, alpha0=alpha0, q_bounds=q_bounds)
        qh, ll1 = _fit_q_mle(x_vec, n_tot, Nb_vec, alpha0=alpha0, q_bounds=q_bounds)
        q_hat[j] = qh
        LRj = max(0.0, 2.0 * (ll1 - ll0))
        LR[j] = LRj
        p = float(chi2.sf(LRj, df=1))
        if (not np.isfinite(p)) or p <= 0.0:
            p = _MIN_PVAL
        pval[j] = p

    tested = np.isfinite(pval)
    qval   = np.full(K, np.nan)
    reject = np.zeros(K, dtype=bool)
    if tested.any():
        r, qv, _, _ = multipletests(pval[tested], alpha=float(fdr_alpha), method=str(fdr_method))
        qval[tested]  = np.maximum(qv, _MIN_PVAL)
        pval[tested]  = np.maximum(pval[tested], _MIN_PVAL)
        reject[tested] = r

    res = pd.DataFrame({
        "feature": list(feature_ids),
        "p_donor": p_donor,
        "q_hat_recipient_center": q_hat,
        "LR": LR,
        "pval": pval,
        "qval_FDR": qval,
        "reject_FDR": reject,
    })
    direction = pd.Series([None]*len(res), dtype="object")
    mask = np.isfinite(res["q_hat_recipient_center"].to_numpy(dtype=float))
    direction.loc[mask] = np.where(
        res.loc[mask, "q_hat_recipient_center"] > res.loc[mask, "p_donor"],
        "up_in_recipient", "down_in_recipient",
    )
    res["direction"] = direction
    return res.sort_values(["pval","LR"], ascending=[True,False]).reset_index(drop=True)
'''

with open('neutrality_test.py', 'w') as f:
    f.write(neutrality_test_source)

print('neutrality_test.py written.')

## 3 · Write the Streamlit app to `app_neutrality.py`

In [ ]:
app_source = '''
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
from neutrality_test import neutrality_lrt

# ── Page config ──────────────────────────────────────────────────────────────
st.set_page_config(page_title="Neutrality LRT", layout="wide")
st.title("🧬 Neutrality Test — Donor → Recipient Transmission")
st.markdown(
    """
    Test whether each feature (taxon / gene) is **neutral** during transmission:
    - **H₀**: recipient center frequency = donor frequency  
    - **H₁**: recipient center frequency is free  
    Uses a likelihood-ratio test (LRT) with Dirichlet–Multinomial marginals
    and Benjamini–Hochberg FDR correction.
    """
)

# ── Sidebar — parameters ──────────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Parameters")
    bottleneck_input = st.text_input(
        "Bottleneck size Nb (scalar or comma-separated per recipient)",
        value="100",
        help="A single number (same Nb for all recipients) or one value per recipient row."
    )
    fdr_alpha   = st.slider("FDR α", 0.01, 0.20, 0.05, 0.01)
    pseudocount = st.number_input("Donor pseudocount", value=1e-6, format="%.2e")
    min_p       = st.number_input("min_p (skip near-zero donor freqs)", value=1e-8, format="%.2e")
    fdr_method  = st.selectbox("FDR method", ["fdr_bh", "fdr_by", "holm", "bonferroni"])
    st.markdown("---")
    st.markdown("**Input format** — CSV files, rows = samples/recipients, columns = features (taxa / genes).")

# ── Data input tabs ───────────────────────────────────────────────────────────
tab_upload, tab_demo = st.tabs(["📂 Upload your data", "🎲 Run demo"])

def parse_bottleneck(text, M):
    parts = [p.strip() for p in text.split(",") if p.strip()]
    if len(parts) == 1:
        return float(parts[0])
    arr = np.array([float(p) for p in parts])
    if len(arr) != M:
        st.error(f"Bottleneck has {len(arr)} values but there are {M} recipients.")
        st.stop()
    return arr

donor_df     = None
recipient_df = None

with tab_upload:
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Donor counts")
        st.caption("One row (or one column) with raw counts per feature.")
        donor_file = st.file_uploader("Upload donor CSV", type="csv", key="donor")
        if donor_file:
            donor_df = pd.read_csv(donor_file, index_col=0)
            st.dataframe(donor_df.head(), use_container_width=True)
    with col2:
        st.subheader("Recipient counts")
        st.caption("Rows = recipients, columns = features (same order as donor).")
        recip_file = st.file_uploader("Upload recipient CSV", type="csv", key="recip")
        if recip_file:
            recipient_df = pd.read_csv(recip_file, index_col=0)
            st.dataframe(recipient_df.head(), use_container_width=True)

with tab_demo:
    st.markdown(
        """
        Generates **synthetic** data: 8 features, 6 recipients.  
        Features A and B are shifted in recipients to simulate non-neutrality.
        """
    )
    if st.button("Generate demo data"):
        rng = np.random.default_rng(42)
        K, M = 8, 6
        features = [f"Feature_{chr(65+i)}" for i in range(K)]
        donor_counts_demo = rng.integers(50, 500, size=K).astype(float)
        donor_df = pd.DataFrame([donor_counts_demo], columns=features, index=["donor"])
        p = donor_counts_demo / donor_counts_demo.sum()
        # shift two features
        p_recip = p.copy()
        p_recip[0] *= 3; p_recip[1] *= 0.2
        p_recip /= p_recip.sum()
        rows = []
        for _ in range(M):
            n = rng.integers(800, 1200)
            rows.append(rng.multinomial(n, p_recip))
        recipient_df = pd.DataFrame(rows, columns=features,
                                    index=[f"recipient_{i+1}" for i in range(M)])
        st.session_state["donor_df"]     = donor_df
        st.session_state["recipient_df"] = recipient_df
        st.success("Demo data generated!")
        col1, col2 = st.columns(2)
        with col1:
            st.write("**Donor**"); st.dataframe(donor_df, use_container_width=True)
        with col2:
            st.write("**Recipients**"); st.dataframe(recipient_df, use_container_width=True)

# pull demo data into local vars if present
if donor_df is None and "donor_df" in st.session_state:
    donor_df     = st.session_state["donor_df"]
if recipient_df is None and "recipient_df" in st.session_state:
    recipient_df = st.session_state["recipient_df"]

# ── Run test ──────────────────────────────────────────────────────────────────
if donor_df is not None and recipient_df is not None:
    st.markdown("---")
    if st.button("▶ Run neutrality LRT", type="primary"):
        with st.spinner("Running LRT …"):
            # flatten donor to 1-D series
            if donor_df.shape[0] == 1:
                donor_series = donor_df.iloc[0]
            elif donor_df.shape[1] == 1:
                donor_series = donor_df.iloc[:, 0]
            else:
                donor_series = donor_df.iloc[0]
                st.warning("Donor has multiple rows; using the first row.")

            M = recipient_df.shape[0]
            Nb = parse_bottleneck(bottleneck_input, M)

            results = neutrality_lrt(
                donor_counts=donor_series,
                recipient_counts=recipient_df,
                bottleneck=Nb,
                pseudocount=pseudocount,
                min_p=min_p,
                fdr_alpha=fdr_alpha,
                fdr_method=fdr_method,
            )
            st.session_state["results"] = results

    if "results" in st.session_state:
        results = st.session_state["results"]

        # ── Summary metrics ────────────────────────────────────────────────
        n_tested   = results["pval"].notna().sum()
        n_rejected = results["reject_FDR"].sum()
        m1, m2, m3 = st.columns(3)
        m1.metric("Features tested", int(n_tested))
        m2.metric(f"Rejected (FDR {int(fdr_alpha*100)}%)", int(n_rejected))
        m3.metric("Not rejected (neutral)", int(n_tested - n_rejected))

        # ── Results table ──────────────────────────────────────────────────
        st.subheader("Results table")
        display_cols = ["feature","p_donor","q_hat_recipient_center",
                        "LR","pval","qval_FDR","reject_FDR","direction"]
        styled = results[display_cols].style.format(
            {"p_donor":"{:.4f}","q_hat_recipient_center":"{:.4f}",
             "LR":"{:.3f}","pval":"{:.2e}","qval_FDR":"{:.2e}"}
        ).apply(
            lambda col: ["background-color: #ffd6d6" if v else "" for v in col],
            subset=["reject_FDR"]
        )
        st.dataframe(styled, use_container_width=True)

        # download
        csv_bytes = results.to_csv(index=False).encode()
        st.download_button("⬇ Download results CSV", csv_bytes,
                           "neutrality_results.csv", "text/csv")

        # ── Plots ──────────────────────────────────────────────────────────
        st.subheader("Visualisations")
        plot_tab1, plot_tab2, plot_tab3 = st.tabs(
            ["p-value distribution", "Volcano (LR vs –log10 p)", "Freq shift"]
        )

        with plot_tab1:
            fig, ax = plt.subplots(figsize=(7, 3))
            pv = results["pval"].dropna()
            ax.hist(pv, bins=20, color="steelblue", edgecolor="white")
            ax.axvline(fdr_alpha, color="red", linestyle="--", label=f"α={fdr_alpha}")
            ax.set_xlabel("p-value"); ax.set_ylabel("Count")
            ax.set_title("p-value histogram"); ax.legend()
            plt.tight_layout()
            st.pyplot(fig, use_container_width=True)

        with plot_tab2:
            fig, ax = plt.subplots(figsize=(7, 4))
            sub = results.dropna(subset=["pval","LR"])
            colors = ["#e74c3c" if r else "#95a5a6" for r in sub["reject_FDR"]]
            ax.scatter(sub["LR"], -np.log10(sub["pval"]), c=colors, alpha=0.8, edgecolors="none")
            for _, row in sub[sub["reject_FDR"]].iterrows():
                ax.annotate(str(row["feature"]), (row["LR"], -np.log10(row["pval"])),
                            fontsize=7, ha="left", va="bottom")
            ax.set_xlabel("Likelihood-ratio statistic")
            ax.set_ylabel("-log₁₀(p-value)")
            ax.set_title("Volcano plot")
            from matplotlib.lines import Line2D
            legend_elements = [
                Line2D([0],[0], marker="o", color="w", markerfacecolor="#e74c3c", label="Rejected"),
                Line2D([0],[0], marker="o", color="w", markerfacecolor="#95a5a6", label="Not rejected"),
            ]
            ax.legend(handles=legend_elements)
            plt.tight_layout()
            st.pyplot(fig, use_container_width=True)

        with plot_tab3:
            fig, ax = plt.subplots(figsize=(8, 4))
            sub = results.dropna(subset=["q_hat_recipient_center"])
            x = np.arange(len(sub))
            ax.bar(x, sub["q_hat_recipient_center"] - sub["p_donor"],
                   color=["#e74c3c" if r else "#3498db" for r in sub["reject_FDR"]])
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set_xticks(x)
            ax.set_xticklabels(sub["feature"].astype(str), rotation=45, ha="right", fontsize=8)
            ax.set_ylabel("q̂ − p_donor  (frequency shift)")
            ax.set_title("Frequency shift per feature (red = rejected)")
            plt.tight_layout()
            st.pyplot(fig, use_container_width=True)
else:
    st.info("Upload donor + recipient CSVs, or click **Run demo** to get started.")
'''

with open('app_neutrality.py', 'w') as f:
    f.write(app_source)

print('app_neutrality.py written.')

## 4 · Preview the app structure

In [ ]:
import ast, pathlib

src = pathlib.Path('app_neutrality.py').read_text()
tree = ast.parse(src)

sections = []
for node in ast.walk(tree):
    if isinstance(node, ast.Call):
        func = node.func
        name = getattr(func, 'attr', None) or getattr(func, 'id', None)
        if name in {'title','header','subheader','markdown','tabs','columns','button','file_uploader','slider','selectbox','metric'}:
            sections.append(f"  st.{name}()")

print("App structure (Streamlit calls found):")
for s in dict.fromkeys(sections):   # deduplicate, preserve order
    print(s)

## 5 · Quick sanity-check: run the core logic in the notebook

In [ ]:
import importlib, sys

# Force reload in case the module was already cached
if 'neutrality_test' in sys.modules:
    del sys.modules['neutrality_test']

from neutrality_test import neutrality_lrt
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
K, M = 8, 6
features = [f"Feature_{chr(65+i)}" for i in range(K)]

donor_counts = rng.integers(50, 500, size=K).astype(float)
p = donor_counts / donor_counts.sum()

p_recip = p.copy()
p_recip[0] *= 3    # Feature_A strongly up in recipients
p_recip[1] *= 0.2  # Feature_B strongly down
p_recip /= p_recip.sum()

rows = [rng.multinomial(rng.integers(800, 1200), p_recip) for _ in range(M)]
recipient_df = pd.DataFrame(rows, columns=features)
donor_series = pd.Series(donor_counts, index=features)

results = neutrality_lrt(
    donor_counts=donor_series,
    recipient_counts=recipient_df,
    bottleneck=100.0,
)

print(results.to_string(index=False))

## 6 · Launch the Streamlit app

Run the cell below. A public URL will appear (via `localtunnel`) so anyone
with internet access can reach the app — no server needed.

> **Alternative**: if you prefer `ngrok`, replace the last block with  
> `!ngrok http 8501` after authenticating your ngrok token.

In [ ]:
import subprocess, threading, time

# Start Streamlit in the background
proc = subprocess.Popen(
    ["streamlit", "run", "app_neutrality.py",
     "--server.port", "8501",
     "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)   # give Streamlit a moment to start
print("Streamlit started on http://localhost:8501")

# ── Option A: localtunnel (no sign-up required) ───────────────────────────────
try:
    subprocess.run(["npx", "--yes", "localtunnel", "--port", "8501"],
                   check=False, timeout=60)
except Exception as e:
    print(f"localtunnel error: {e}")

# If localtunnel isn't available, run this instead (requires Node.js):
#   !npx --yes localtunnel --port 8501
#
# Or with ngrok (requires free token at https://ngrok.com):
#   !ngrok authtoken YOUR_TOKEN
#   !ngrok http 8501

---
### Deploying permanently (no tunnel needed)

| Platform | Steps |
|---|---|
| **Streamlit Community Cloud** | Push repo to GitHub → [share.streamlit.io](https://share.streamlit.io) → Deploy |
| **Hugging Face Spaces** | Create a Space (Streamlit SDK) → upload `app_neutrality.py` + `neutrality_test.py` + `requirements.txt` |
| **Render / Railway** | Connect GitHub repo, set start command `streamlit run app_neutrality.py --server.port $PORT` |

Your `requirements.txt` should contain:
```
streamlit
numpy
pandas
scipy
statsmodels
matplotlib
```